# MOD-12930 — FT.HYBRID vs its subqueries (balanced suite)

FT.HYBRID is timed alongside its two subqueries run standalone (same K/WINDOW, same
reply size). Per (size, depth) cell, the text query is tuned so the SEARCH subquery's
latency is similar to the VSIM subquery's — with heavily skewed subqueries the comparison
says little about concurrency or merge cost. Note the SEARCH numbers are therefore *not*
the native scaling of one fixed text query.

**Matrix:** size (10K/100K/500K) × depth K/W (10/20, 50/100, 250/500) × workers=2 ×
loader (keyspace access) on/off × 4 contenders. Dataset: dbpedia, 512-dim embeddings,
HNSW cosine. Before timing, every configuration must pass a correctness gate: FT.HYBRID's
output must equal an offline fusion of the two raw queries (tie-aware).

Set `REUSE_RESULTS=1` to re-render tables from saved results without re-running.

In [1]:
import json, os
import pandas as pd

if os.environ.get('REUSE_RESULTS') and os.path.exists('results_balanced_full.json'):
    d = json.load(open('results_balanced_full.json'))
    results, gates, meta = d['results'], d['gates'], d['meta']
    print('reusing saved results_balanced_full.json')
else:
    import bench_lib as B
    import balanced_lib as BAL
    results, gates, meta = BAL.run_balanced_full(*B.load_data())
    with open('results_balanced_full.json', 'w') as f:
        json.dump(dict(meta=meta, results=results, gates=gates), f, indent=2, default=str)
    pd.DataFrame(results).to_csv('results_balanced_full.csv', index=False)
    print('saved results_balanced_full.json / .csv')

reusing saved results_balanced_full.json


## Calibration & gates

`balance_ratio` = search p50 / vsim p50 at calibration.

In [2]:
pd.DataFrame(gates)

,size,k,window,matches_mean,balance_ratio,gate_linear,gate_rrf
0,10000,10,20,1192.50000,1.08,16/16,16/16
1,10000,50,100,644.90625,0.88,16/16,16/16
2,10000,250,500,2970.34375,0.98,16/16,16/16
3,100000,10,20,1689.31250,1.07,16/16,16/16
4,100000,50,100,5626.56250,1.03,16/16,16/16
5,100000,250,500,12625.46875,0.92,16/16,16/16
6,500000,10,20,1436.87500,0.93,16/16,16/16
7,500000,50,100,2781.40625,1.02,16/16,16/16
8,500000,250,500,9446.31250,1.02,16/16,16/16


## p50 latency (ms) by contender

In [3]:
df = pd.DataFrame(results)
df.pivot_table(index=['size', 'window', 'fields'], columns='contender',
               values='p50_ms', sort=False).round(2)[
    ['hybrid_linear', 'hybrid_rrf', 'search_branch', 'vsim_branch']]

contender                 hybrid_linear  hybrid_rrf  search_branch  \
size   window fields                                                 
10000  20     none                 0.37        0.34           0.20   
              title+text           0.41        0.40           0.26   
       100    none                 0.54        0.56           0.24   
              title+text           0.63        0.66           0.30   
       500    none                 2.36        2.31           1.04   
              title+text           2.77        2.66           1.01   
100000 20     none                 0.42        0.43           0.25   
              title+text           0.43        0.46           0.35   
       100    none                 0.89        0.90           0.40   
              title+text           1.05        1.03           0.53   
       500    none                 3.16        3.15           1.37   
              title+text           3.54        3.68           1.41   
500000 20     none                 0.41        0.43           0.30   
              title+text           0.45        0.50           0.41   
       100    none                 0.94        0.93           0.54   
              title+text           1.04        1.06           0.60   
       500    none                 3.53        3.49           1.69   
              title+text           3.91        3.95           1.76   

contender                 vsim_branch  
size   window fields                   
10000  20     none               0.16  
              title+text         0.22  
       100    none               0.24  
              title+text         0.36  
       500    none               0.97  
              title+text         0.90  
100000 20     none               0.23  
              title+text         0.27  
       100    none               0.43  
              title+text         0.51  
       500    none               1.46  
              title+text         1.52  
500000 20     none               0.23  
              title+text         0.32  
       100    none               0.49  
              title+text         0.57  
       500    none               1.66  
              title+text         1.71

## Merger sweep

Same contenders with window, dataset size and text selectivity varied independently
(match-fraction 1% / 10% / 50% of the corpus; the calibrated matrix above cannot separate
these axes). Raw p50 per contender.

In [4]:
if os.environ.get('REUSE_RESULTS') and os.path.exists('results_merger_sweep.json'):
    sw = json.load(open('results_merger_sweep.json'))
    sweep_results = sw['results']
    print('reusing saved results_merger_sweep.json')
else:
    import bench_lib as B
    from merger_sweep import run_sweep
    sweep_results, _ = run_sweep(*B.load_data())

sdf = pd.DataFrame(sweep_results)
sdf.pivot_table(index=['size', 'match_frac'], columns='window',
                values=['hybrid_p50_ms', 'search_p50_ms', 'vsim_p50_ms']).round(2)

reusing saved results_merger_sweep.json


hybrid_p50_ms               search_p50_ms                \
window                      20     100    500           20     100    500   
size   match_frac                                                           
10000  0.01                0.23   0.57   1.56          0.16   0.16   0.15   
       0.10                0.36   0.61   1.83          0.20   0.28   0.38   
       0.50                0.63   0.99   2.53          0.56   0.66   1.23   
100000 0.01                0.37   0.79   2.36          0.24   0.32   0.51   
       0.10                0.98   1.34   3.02          0.56   0.74   1.07   
       0.50                5.26   6.25   9.25          4.69   5.25   7.04   
500000 0.01                1.00   1.41   3.49          0.55   0.88   1.33   
       0.10                6.72   7.40  11.06          5.01   4.02   4.41   
       0.50               37.24  35.94  40.46         31.64  35.27  35.01   

                  vsim_p50_ms              
window                    20    100   500  
size   match_frac                          
10000  0.01              0.20  0.26  0.91  
       0.10              0.16  0.25  0.95  
       0.50              0.15  0.24  0.83  
100000 0.01              0.21  0.44  1.48  
       0.10              0.22  0.42  1.47  
       0.50              0.22  0.42  1.44  
500000 0.01              0.27  0.50  1.83  
       0.10              0.24  0.78  1.98  
       0.50              0.26  0.52  1.73